# Diffusion-ES debugger

Inspect one scenario end to end:

1. **Pick** a tfrecord + scenario index.
2. **Run** closed-loop diffusion ES (same loop as `simulation/runner.py`) and watch the video.
3. **Anchor** on a replan step near the bad behaviour: see every ES iteration (candidates, scores, elites, reward terms) and the world-model futures the scorer used.
4. **Experiment**: swap the reward and/or the acquisition (elite selection) on top of the same diffusion policy, re-plan at the anchor, and re-run the closed loop from that anchor to see if the failure is recoverable.

Logic lives in `notebooks/es_debug.py`. Run on the cluster with the same env as `slurm_es_fail1500_wm_mf.sh` (on a GPU node: see the `jupyter-tunnel` skill).

In [ ]:
import os
# Torch WM + JAX planner share one GPU; must be set before JAX is imported.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import es_debug as esd

## 1. Configuration

In [ ]:
# ---- scenario ---------------------------------------------------------------
SUBSET_DIR = "/zfsauton/scratch/sbellad/Diffusion_ES_sim/es_failure_subset_n1500_seed0"
SHARD = "training_tfexample.tfrecord-00172-of-01000"   # <- tfrecord file name (the 'tfrecord' field of a result json)
SCENARIO_IDX = 171                                     # <- record index inside it ('scenario_idx' field)
TFRECORD_PATH = f"{SUBSET_DIR}/tfrecords/training/{SHARD}"
TARGET_JSON = f"{SUBSET_DIR}/target_scenarios.json"               # optional: lists failure indices per shard
LANE_GRAPH_DIR = "/zfsauton/scratch/mineuih/waymax_rs/lane_graphs"

# ---- models -----------------------------------------------------------------
DIFFUSION_CKPT = "/zfsauton/scratch/sbellad/train_diffusion/diffusion_baseline_baseline_20260810_144559/epoch_0500"
WM_CKPT = "/zfsauton/scratch/ishanj/world_model/ckpts_multifuture_gaussian/multiFuture_GaussianWorldModel_20261005_092950/latest.pt"
WM_BACKEND = "multifuture"
WM_PLANNING_MODE = "expected_independent"
OTHER_AGENT_FORECAST = "wm"   # wm | cv | gt

# ---- ES / sim (defaults = slurm_es_fail1500_wm_mf.sh) -----------------------
ES = dict(population_size=64, elite_size=8, num_iterations=4, resample_timesteps=5)
START_TIMESTEP = 10
REPLAN_INTERVAL = 10
SIM_AGENTS = "log"            # log | idm (how non-ego agents move in the evaluation rollout)
SEED = 0

In [ ]:
# Optional: which scenario indices of this shard are in the failure subset.
if Path(TARGET_JSON).exists():
    targets = json.load(open(TARGET_JSON))
        print(f"{len(targets)} shards in subset; e.g. {list(targets)[:3]}")
    print(f"failure indices for {SHARD}: {targets.get(SHARD, 'shard not in subset')}")

## 2. Load models and scenario

In [ ]:
policy, preprocess_cfg = esd.load_diffusion_policy(DIFFUSION_CKPT, seed=SEED)
wm = (
    esd.load_world_model(WM_CKPT, backend=WM_BACKEND, planning_mode=WM_PLANNING_MODE)
    if OTHER_AGENT_FORECAST == "wm" else None
)

In [ ]:
scenario = esd.load_scenario(TFRECORD_PATH, SCENARIO_IDX, lane_graph_dir=LANE_GRAPH_DIR)
print(scenario.name, "ego idx:", scenario.ego_idx, "goal:", np.asarray(scenario.goal_xy_b2[0]), "goal t:", scenario.goal_t)

## 3. Baseline closed-loop run

`DebugESPlanner` with `production_reward()` (collision × offroad) and `topk` is the production ES. Every replan step stores the full ES trace.

In [ ]:
baseline_planner = esd.DebugESPlanner(
    policy, preprocess_cfg,
    other_agent_forecast=OTHER_AGENT_FORECAST, world_model_forecaster=wm,
    reward=esd.production_reward(), acquisition=esd.topk, label="baseline",
    **ES,
)
base = esd.run_closed_loop(
    baseline_planner, scenario,
    start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
    sim_agents=SIM_AGENTS, seed=SEED,
)

In [ ]:
# Scrub with the slider. Dashed blue = plan being executed, ego turns red on overlap/offroad.
esd.animate_closed_loop(base, view_m=40, show_plans=True, show_elites=True, show_ids=False)

In [ ]:
# Use this to pick the anchor: failure spans are shaded, ES best/mean score per replan.
esd.plot_timeline(base);

## 4. Inspect ES at an anchor

Set `ANCHOR_T` to a time just inside the bad window; it is snapped to the replan step whose plan was being executed at that time. Use `anchor - REPLAN_INTERVAL` to look at the decision before.

In [ ]:
ANCHOR_T = 40   # <- set me
anchor = base.anchor_for(ANCHOR_T)
trace = base.traces[anchor]
state_at_anchor = base.snapshots[anchor]
print(f"anchor replan step = {anchor}; chosen plan score = {trace.best_score:.3f}")
pd.DataFrame(trace.summary())

In [ ]:
# Candidates per ES iteration, coloured by score. Bold = elites picked by the acquisition, red = final plan.
esd.plot_es_iterations(trace, state_at_anchor, ego_idx=scenario.ego_idx, view_m=35);

In [ ]:
esd.plot_score_evolution([trace]);

In [ ]:
# Top candidates of the final pool with every reward term; change iteration= to look earlier.
esd.candidates_table(trace, iteration=-1, top=20)

### World-model futures at the anchor

All K modes per agent (line width/alpha ~ mode probability) against the logged future (black dashed). Agents with nonzero collision probability against the chosen plan are orange.

In [ ]:
esd.plot_wm_forecast(trace, state_at_anchor, ego_idx=scenario.ego_idx, view_m=35, min_mode_prob=0.05);
display(esd.risky_agents(trace))

In [ ]:
# Step through the horizon: chosen ego plan vs WM top-1 boxes vs logged boxes.
# Pass candidate=trace.iterations[-1].trajectories[k] to inspect another candidate.
esd.animate_wm_forecast(trace, state_at_anchor, ego_idx=scenario.ego_idx, view_m=35, stride=2)

In [ ]:
# Sanity check: production Scorer.compute_score should match the recorded scores for production_reward().
last = trace.iterations[-1]
prod = np.asarray(baseline_planner.scorer.compute_score(
    trajectories=last.trajectories, sim_state=state_at_anchor, timestep=anchor, world_idx=0,
))
print("max |production - recorded| =", float(np.abs(prod - last.scores).max()))

## 5. Experiments: change the reward / acquisition

Building blocks in `es_debug`:

* **Reward terms** (`ctx -> [P]` in [0, 1]): `collision`, `wm_collision(mode)`, `offroad`, `direction`, `tl_violation`, `progress`, `goal(max_dist_m)`, `clearance(max_dist_m)`, `target_speed(v)`, `comfort(max_accel)`. Combine them with `Reward(multiplicative={...}, additive={name: (term, weight)})`, which uses the same algebra as `Scorer.compute_score`.
* **Acquisitions** (`(scores, trajs, n_elite, rng) -> indices`): `topk`, `boltzmann(T)`, `diverse_topk(min_dist_m)`, `epsilon_greedy(eps)`.
* `planner.variant(reward=..., acquisition=..., elite_size=..., num_iterations=..., label=...)` reuses the compiled diffusion functions. Changing `population_size` / `resample_timesteps` rebuilds and re-jits.

Write your own term as a plain function of `esd.RewardContext` (fields: `trajectories`, `ego_boxes`, `object_trajectories`, `object_masks`, `sim_state`, `timestep`, `scorer`, `world_model_forecaster`, `goal_xy`).

In [ ]:
# Example custom term: penalise plans that stop (low min speed) -- edit freely.
def keep_moving(ctx: esd.RewardContext, min_speed=1.0):
    speed = np.hypot(np.asarray(ctx.trajectories[..., 3]), np.asarray(ctx.trajectories[..., 4]))
    return np.clip(speed.min(axis=1) / min_speed, 0, 1).astype(np.float32)

variants = {
    "worst_case_wm": baseline_planner.variant(
        label="worst_case_wm",
        reward=esd.Reward(multiplicative={"collision": esd.wm_collision("worst_case_independent"), "offroad": esd.offroad}),
    ),
    "shaped": baseline_planner.variant(
        label="shaped",
        reward=esd.Reward(
            multiplicative={"collision": esd.collision, "offroad": esd.offroad},
            additive={"progress": (esd.progress, 1.0), "clearance": (esd.clearance(8.0), 1.0), "comfort": (esd.comfort(), 0.5)},
        ),
    ),
    "diverse": baseline_planner.variant(label="diverse", acquisition=esd.diverse_topk(3.0)),
    "boltzmann": baseline_planner.variant(label="boltzmann", acquisition=esd.boltzmann(0.1), num_iterations=6),
}

### 5a. Re-plan only at the anchor (same state, same seed)

In [ ]:
anchor_traces = {"baseline": trace}
for name, planner in variants.items():
    anchor_traces[name] = esd.replan_at(planner, base, anchor, seed=SEED)

pd.DataFrame([
    {"variant": k, "best_score": t.best_score, **t.summary()[-1]} for k, t in anchor_traces.items()
]).set_index("variant")

In [ ]:
esd.plot_score_evolution(list(anchor_traces.values()));

In [ ]:
VARIANT = "shaped"   # <- inspect one
vt = anchor_traces[VARIANT]
esd.plot_es_iterations(vt, state_at_anchor, ego_idx=scenario.ego_idx, view_m=35);
display(esd.candidates_table(vt, top=10))

### 5b. Closed loop with the variant

`resume_at=anchor` replays the baseline's plans up to the anchor and switches to the variant from there. This tests whether the failure can be recovered from that state. Drop `resume_from`/`resume_at` to run the variant from the start.

In [ ]:
runs = {}
for name, planner in variants.items():
    print(f"--- {name}")
    runs[name] = esd.run_closed_loop(
        planner, scenario,
        start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
        sim_agents=SIM_AGENTS, seed=SEED,
        resume_from=base, resume_at=anchor, verbose=False,
    )
    print(runs[name].report())

pd.DataFrame([
    {"variant": r.label, "success": r.success, "goal_reached": r.goal_reached,
     "overlap": r.overlap, "offroad": r.offroad, "min_goal_dist_m": r.min_goal_distance_m}
    for r in [base, *runs.values()]
]).set_index("variant")

In [ ]:
esd.plot_timeline(base, compare=list(runs.values()));

In [ ]:
esd.animate_closed_loop(runs[VARIANT], view_m=40, show_plans=True, show_elites=True,
                        t_range=(max(0, anchor - 10), 90))